In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 原管状块末步写入：独立完整视频验证（准备版）

固定两个已有holdout组合：白天鹅/20261001、蓝色缆车/20261002。内容和seed与本路线4个dev不重合；
这些名单曾用于另一反演候选的holdout和裁剪；这里只是相对本tube/state开发集的预先固定跨内容/种子验证。方法和参数冻结，未在本地运行真实模型或重评分。
每case OFF/LAST_A/B，共6视频24个真实RGB接收phase encode，不运行裁剪/多步。
共享prefix0..48与step49 CFG/native history；LAST使用原margin1完整clean投影及v−u/sigma。
CPU P(OFF)只作终态数值等价参照，不产生TERMINAL独立视频。
原1760块/state×polarity×sync编码、五mode盲接收、CRF18/yuv420p媒体设置不变；无调参/阈值。
每case100Transformer/52native，全轮200/104、6decode/6MP4/24encode。

每组两消息共4marked、各mode固定分母；全部失败/缺失保留。接收后才计算best_correct−best_other；
缺候选为null，OFF仅排名和score0−score1；不是FPR或逐bit payload恢复。
4个phase来自完整收到视频内部起点（181/177/177/177帧），不是未知时间裁剪攻击。
摘要含消息间隔、support/phase coverage、OFF排名、质量、实际控制量和终态等价。
请运行后观看各case的 `received_videos/OFF.mp4`、`LAST_A.mp4`、`LAST_B.mp4`，
检查内容、伪影与连贯性；质量指标仅诊断，不设强制门槛，也尚未完成用户观看检查。

独立输出 `MyDrive/Video-WM/FlowTubeStateHoldout/flow_tube_state_holdout_<UTC>`。
保存source/config/码本/共享prefix/history/终态/视频/接收张量及哈希，生成与VAE子进程隔离。
只完成CPU/fake/static验证，科学结果待用户Colab执行。源码SHA：0797064cc5f1166170ed5c229e18defd681d8185。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = '0797064cc5f1166170ed5c229e18defd681d8185'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'flow_tube_state_holdout_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/FlowTubeStateHoldout') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.flow_tube_state_holdout_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({k:result.get(k) for k in ('status','video_denominator','receiver_encode_denominator','fixed_calls','actual_calls_observed','recovery_summary','message_and_coverage_summary','equivalence_summary','control_summary','quality_summary','OFF_summary')}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
